# Utils - QB - IndexRegularizer

Ce notebook illustre et vérifie le comportement de la classe
`IndexRegularizer` (`tsforecast/frequency/regularizer.py`), qui détecte puis
comble les trous d'un index temporel (série temporelle ou panel) en le
réindexant sur une grille régulière (`pd.date_range`), pour obtenir un index
sans trous avant les étapes d'imputation/agrégation ultérieures.

La classe expose **deux méthodes publiques** : `is_regular()` et
`regularize()`. Le module expose en plus **deux fonctions de commodité** du
même nom, wrappers directs d'une instance singleton (`_regularizer =
IndexRegularizer()`, cf. §5). Les méthodes `_prepare_data`, `_is_regular_ts`,
`_is_regular_panel`, `_resolve_frequency_ts`, `_validate_consistent_positions`,
`_regularize_ts` et `_regularize_panel` sont des auxiliaires privés ; leur
comportement n'est illustré ici qu'au travers de ses effets observables sur
les deux méthodes publiques.

**⚠️ Avertissement (détaillé en §4.6)** : l'exploration ci-dessous met en
évidence un bug reproductible et significatif de `regularize()`. Dès que la
fréquence source est **ancrée** (position début/fin de période, ex. `MS`,
`QS`, `YS` — le cas de la quasi-totalité des données macroéconomiques du
projet) et que l'index n'est pas *déjà* parfaitement régulier, la détection
de fréquence de repli peut renvoyer le code **sans position** (`M`, `Q`, `Y`
au lieu de `MS`, `QS`, `YS`). `pd.date_range` construit alors un nouvel index
qui ne coïncide avec **aucune** des dates d'origine, et `reindex` renvoie un
DataFrame de la bonne forme mais **entièrement `NaN`** — silencieusement,
sans erreur ni avertissement. Le bug est reproduit aussi bien sur les jeux de
données de référence du projet (`df_timeseries`/`df_panel`, notebook 3) que
sur le contenu, jamais exécuté, de `notebooks/test_regularizer.ipynb`.

## 1 - Import et instanciation

In [ ]:
# Importation des modules
import warnings

import numpy as np
import pandas as pd

# Classe et fonctions testées
from tsforecast.frequency.regularizer import IndexRegularizer, is_regular, regularize
from tsforecast.utils.frequency.utils import detect_index_frequency

# Configuration de l'affichage
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
warnings.filterwarnings('ignore')

# Instanciation
regularizer = IndexRegularizer()

print("IndexRegularizer instancié avec succès !")

## 2 - Jeux de données

### 2.1 - Reprise des jeux de données de `3 - QB - Panel a frequences mixtes heterogene.ipynb`

Les deux fonctions génératrices sont recopiées telles quelles (convention déjà
suivie dans `target_frequency_validator.ipynb`, `frequency_aligner.ipynb`,
etc. : aucune fonction partagée n'existe entre notebooks dans ce projet) pour
obtenir `df_timeseries` (indicateurs macro mensuels/trimestriels/annuels,
avec une variable annuelle démarrant avant la grille mensuelle) et `df_panel`
(mêmes indicateurs pour 3 pays, périodes couvertes et fréquences de
publication hétérogènes par entité).

In [ ]:
# Fonction de création de séries temporelles (recopiée depuis le notebook 3)
def create_timeseries_dataset(
    start_date: str = '2018-01-01',
    end_date: str = '2024-07-01',
    annual_start_date: str = '2015-01-01',
    seed: int = 42
) -> pd.DataFrame:
    """Create a realistic macroeconomic time series dataset with mixed frequencies.

    Args:
        start_date: Start date for the monthly variables of the dataset.
        end_date: End date for the dataset.
        annual_start_date: Start date for the annual trade balance series, earlier
            than `start_date` so that the resulting temporal index is irregular.
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with mixed-frequency macroeconomic indicators.
    """
    np.random.seed(seed)

    dates = pd.date_range(start=start_date, end=end_date, freq='MS')
    n_periods = len(dates)

    df = pd.DataFrame(index=dates)
    df.index.name = 'date'

    # Production industrielle (mensuelle)
    trend = np.linspace(100, 115, n_periods)
    seasonal = 3 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
    noise = np.random.normal(0, 1.5, n_periods)
    df['production_industrielle'] = trend + seasonal + noise

    # Inflation mensuelle (IPC)
    inflation_trend = np.linspace(1.2, 2.8, n_periods)
    inflation_noise = np.random.normal(0, 0.3, n_periods)
    df['inflation_ipc'] = np.clip(inflation_trend + inflation_noise, 0.5, 5.0)

    # Taux de chômage (mensuel)
    chomage_trend = np.concatenate([
        np.linspace(8.5, 7.0, n_periods // 3),
        np.linspace(7.0, 9.5, n_periods // 3),
        np.linspace(9.5, 7.5, n_periods - 2 * (n_periods // 3))
    ])
    chomage_noise = np.random.normal(0, 0.2, n_periods)
    df['taux_chomage'] = np.clip(chomage_trend + chomage_noise, 4.0, 15.0)

    # PIB trimestriel
    pib_base = 2500
    pib_growth_quarterly = 0.5
    df['pib_trimestriel'] = np.nan
    quarter_start_months = [1, 4, 7, 10]
    quarter_idx = 0
    for i, date in enumerate(dates):
        if date.month in quarter_start_months:
            growth = pib_growth_quarterly + np.random.normal(0, 0.3)
            df.loc[date, 'pib_trimestriel'] = pib_base * (1 + growth / 100) ** quarter_idx
            quarter_idx += 1

    # Balance commerciale annuelle : historique antérieur à la grille mensuelle
    annual_dates = pd.date_range(start=annual_start_date, end=end_date, freq='YS')
    df = df.reindex(df.index.union(annual_dates))
    df.index.name = 'date'

    df['balance_commerciale_annuelle'] = np.nan
    for date in annual_dates:
        year_factor = (date.year - 2018)
        base_balance = -25 + year_factor * 3 + np.random.normal(0, 5)
        df.loc[date, 'balance_commerciale_annuelle'] = base_balance

    # Délais de publication
    df.loc[df.index[-1], 'inflation_ipc'] = np.nan
    df.loc[df.index[-1], 'taux_chomage'] = np.nan

    pib_available = df[df['pib_trimestriel'].notna()].index
    if len(pib_available) > 0:
        df.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

    bc_available = df[df['balance_commerciale_annuelle'].notna()].index
    if len(bc_available) > 0:
        df.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

    # Historique limité de la production industrielle
    mask_before_2019 = df.index < '2019-01-01'
    df.loc[mask_before_2019, 'production_industrielle'] = np.nan

    return df


# Création du jeu de données de séries temporelles
df_timeseries = create_timeseries_dataset()

print(f"df_timeseries : {df_timeseries.shape}, période {df_timeseries.index.min().date()} à {df_timeseries.index.max().date()}")
df_timeseries.head(5)

In [ ]:
# Fonction de création d'un jeu de données de panel (recopiée depuis le notebook 3)
def create_panel_dataset(seed: int = 42) -> pd.DataFrame:
    """Create a realistic macroeconomic panel dataset with mixed frequencies.

    Each entity has its own coverage period (start/end dates) and its own
    publication frequency for the public spending indicator, to simulate a
    heterogeneous panel across entities. The annual trade balance series
    also starts earlier than the other variables for each entity, making
    each entity's temporal index irregular.

    Args:
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with MultiIndex (country, date) and mixed-frequency indicators.
    """
    np.random.seed(seed)

    countries = {
        'France': {
            'pib_base': 2800, 'inflation_base': 1.5, 'chomage_base': 8.0, 'depenses_base': 55.0,
            'start_date': '2018-01-01', 'end_date': '2024-07-01', 'prod_ind_start': '2018-06-01',
            'depenses_frequency': 'annuelle', 'annual_start_date': '2015-01-01'
        },
        'Allemagne': {
            'pib_base': 3500, 'inflation_base': 1.2, 'chomage_base': 5.5, 'depenses_base': 45.0,
            'start_date': '2018-07-01', 'end_date': '2024-04-01', 'prod_ind_start': '2019-01-01',
            'depenses_frequency': 'trimestrielle', 'annual_start_date': '2016-01-01'
        },
        'Italie': {
            'pib_base': 2200, 'inflation_base': 1.8, 'chomage_base': 10.5, 'depenses_base': 50.0,
            'start_date': '2019-01-01', 'end_date': '2024-07-01', 'prod_ind_start': '2019-06-01',
            'depenses_frequency': 'annuelle', 'annual_start_date': '2016-01-01'
        }
    }

    all_data = []
    for country, params in countries.items():
        np.random.seed(seed + hash(country) % 1000)

        dates = pd.date_range(start=params['start_date'], end=params['end_date'], freq='MS')
        n_periods = len(dates)

        df_country = pd.DataFrame(index=dates)
        df_country['country'] = country

        trend = np.linspace(100, 112 + np.random.uniform(-3, 3), n_periods)
        seasonal = 2.5 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
        noise = np.random.normal(0, 1.2, n_periods)
        df_country['production_industrielle'] = trend + seasonal + noise
        prod_start = pd.Timestamp(params['prod_ind_start'])
        df_country.loc[df_country.index < prod_start, 'production_industrielle'] = np.nan

        infl_trend = np.linspace(
            params['inflation_base'], params['inflation_base'] + np.random.uniform(0.5, 2.0), n_periods
        )
        infl_noise = np.random.normal(0, 0.25, n_periods)
        df_country['inflation_ipc'] = np.clip(infl_trend + infl_noise, 0.3, 6.0)

        chomage_base = params['chomage_base']
        chomage_evolution = np.concatenate([
            np.linspace(chomage_base, chomage_base - 1, n_periods // 3),
            np.linspace(chomage_base - 1, chomage_base + 2, n_periods // 3),
            np.linspace(chomage_base + 2, chomage_base + 0.5, n_periods - 2 * (n_periods // 3))
        ])
        chomage_noise = np.random.normal(0, 0.15, n_periods)
        df_country['taux_chomage'] = np.clip(chomage_evolution + chomage_noise, 2.5, 15.0)

        df_country['pib_trimestriel'] = np.nan
        quarter_end_months = [1, 4, 7, 10]
        quarter_idx = 0
        for date in dates:
            if date.month in quarter_end_months:
                growth = 0.4 + np.random.normal(0, 0.35)
                df_country.loc[date, 'pib_trimestriel'] = params['pib_base'] * (1 + growth / 100) ** quarter_idx
                quarter_idx += 1

        # Dépenses publiques : fréquence annuelle ou trimestrielle selon le pays
        df_country['depenses_publiques_pib'] = np.nan
        if params['depenses_frequency'] == 'annuelle':
            publication_months = [1]
        else:
            publication_months = [1, 4, 7, 10]

        depenses_idx = 0
        for date in dates:
            if date.month in publication_months:
                value = params['depenses_base'] + 0.1 * depenses_idx + np.random.normal(0, 1.0)
                df_country.loc[date, 'depenses_publiques_pib'] = value
                depenses_idx += 1

        # Balance commerciale annuelle : historique antérieur à la grille mensuelle du pays
        annual_dates = pd.date_range(start=params['annual_start_date'], end=params['end_date'], freq='YS')
        df_country = df_country.reindex(df_country.index.union(annual_dates))
        df_country['country'] = country

        df_country['balance_commerciale_annuelle'] = np.nan
        for date in annual_dates:
            year_factor = (date.year - 2018)
            base = -20 + np.random.uniform(-10, 10) + year_factor * 2
            df_country.loc[date, 'balance_commerciale_annuelle'] = base

        # Délais de publication
        df_country.loc[df_country.index[-1], 'inflation_ipc'] = np.nan
        df_country.loc[df_country.index[-1], 'taux_chomage'] = np.nan

        pib_available = df_country[df_country['pib_trimestriel'].notna()].index
        if len(pib_available) > 0:
            df_country.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

        bc_available = df_country[df_country['balance_commerciale_annuelle'].notna()].index
        if len(bc_available) > 0:
            df_country.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

        depenses_available = df_country[df_country['depenses_publiques_pib'].notna()].index
        if len(depenses_available) > 0:
            df_country.loc[depenses_available[-1], 'depenses_publiques_pib'] = np.nan

        all_data.append(df_country)

    df_panel = pd.concat(all_data, ignore_index=False)
    df_panel = df_panel.reset_index().rename(columns={'index': 'date'})
    df_panel = df_panel.set_index(['country', 'date'])
    df_panel = df_panel.sort_index()

    return df_panel


# Création du jeu de données de panel
df_panel = create_panel_dataset()

print(f"df_panel : {df_panel.shape}")
df_panel.loc['France'].head(5)

### 2.2 - Jeux de données ciblés supplémentaires

En complément de `df_timeseries`/`df_panel`, quelques jeux de données minimaux
et contrôlés sont construits pour isoler précisément chaque comportement
(fréquence propre connue, position ancrée ou non, entités de fréquences
différentes, etc.), sans le bruit des nombreuses colonnes des jeux réalistes.

In [ ]:
# Série mensuelle (MS) parfaitement régulière, sans trou
ts_regular_ms = pd.Series(
    range(24), index=pd.date_range('2020-01-01', periods=24, freq='MS'), dtype=float, name='v'
)

# Même série, avec le mois de juin 2020 manquant (un seul trou, fréquence ancrée 'MS')
ts_gap_ms = ts_regular_ms.drop(pd.Timestamp('2020-06-01'))

# Série journalière (fréquence 'D', sans position) avec un trou
ts_gap_daily = pd.Series(
    range(19), index=pd.date_range('2020-01-01', periods=20, freq='D').delete(10), dtype=float, name='v'
)

# Dates totalement irrégulières (aucune fréquence détectable)
ts_no_freq = pd.Series(
    range(5),
    index=pd.to_datetime(['2020-01-01', '2020-01-05', '2020-03-11', '2020-08-02', '2021-01-30']),
    dtype=float, name='v',
)

# Horodatages dupliqués
ts_duplicated = pd.Series([1.0, 2.0, 3.0], index=pd.to_datetime(['2020-01-01', '2020-01-01', '2020-02-01']))

print("ts_regular_ms  :", len(ts_regular_ms), "obs -", ts_regular_ms.index.inferred_freq)
print("ts_gap_ms      :", len(ts_gap_ms), "obs (juin 2020 manquant)")
print("ts_gap_daily   :", len(ts_gap_daily), "obs -", ts_gap_daily.index.inferred_freq)
print("ts_no_freq     :", len(ts_no_freq), "obs, dates aléatoires")
print("ts_duplicated  :", ts_duplicated.index.tolist())

In [ ]:
# Panel minimal à deux entités de fréquences NATIVES différentes :
# A publiée mensuellement (MS), B publiée trimestriellement (QS) — chacune régulière
# individuellement, mais globalement hétérogène.
idx_a = pd.MultiIndex.from_arrays(
    [['A'] * 12, pd.date_range('2020-01-01', periods=12, freq='MS')], names=['entity', 'date']
)
idx_b = pd.MultiIndex.from_arrays(
    [['B'] * 4, pd.date_range('2020-01-01', periods=4, freq='QS')], names=['entity', 'date']
)
panel_mixed_native_freq = pd.concat([
    pd.DataFrame({'v': range(12)}, index=idx_a, dtype=float),
    pd.DataFrame({'v': range(100, 104)}, index=idx_b, dtype=float),
])

# Panel avec une entité C ne comportant qu'UNE seule observation (fréquence indétectable)
idx_c = pd.MultiIndex.from_tuples([('C', pd.Timestamp('2020-06-01'))], names=['entity', 'date'])
panel_single_obs_entity = pd.concat([
    panel_mixed_native_freq,
    pd.DataFrame({'v': [999.0]}, index=idx_c),
])

print("panel_mixed_native_freq :", panel_mixed_native_freq.shape)
panel_mixed_native_freq

## 3 - `is_regular(data, time_col=None, panel_cols=None, per_entity=False)`

Vérifie que l'index temporel est régulier : pour une série temporelle,
`pd.infer_freq(index) is not None` ; pour un panel, chaque entité est testée
individuellement, et le résultat global (`per_entity=False`) n'est `True` que
si **toutes** les entités sont régulières **et** partagent la **même**
fréquence.

### 3.1 - Séries temporelles : cas régulier / irrégulier

In [ ]:
# Série mensuelle sans trou -> True
print("ts_regular_ms  :", regularizer.is_regular(ts_regular_ms))

# Un seul mois manquant suffit à casser la régularité -> False
print("ts_gap_ms      :", regularizer.is_regular(ts_gap_ms))

# df_timeseries (notebook 3) : variable annuelle antérieure à la grille mensuelle -> False
print("df_timeseries  :", regularizer.is_regular(df_timeseries))

# Sous-période purement mensuelle de df_timeseries (sans les dates annuelles isolées) -> True
ts_timeseries_clean_subset = df_timeseries.loc['2019-01-01':'2024-06-01', ['production_industrielle']]
print("sous-période propre :", regularizer.is_regular(ts_timeseries_clean_subset))

### 3.2 - `time_col` : équivalent en colonne plutôt qu'en index

`time_col` déplace la colonne indiquée en index avant le test ; le résultat
est identique à celui obtenu avec la donnée déjà indexée.

In [ ]:
df_timeseries_flat = df_timeseries.reset_index()  # 'date' redevient une colonne

print("Indexé (référence)     :", regularizer.is_regular(df_timeseries))
print("Via time_col='date'    :", regularizer.is_regular(df_timeseries_flat, time_col='date'))

### 3.3 - Panel : résultat global (`per_entity=False`) vs par entité (`per_entity=True`)

In [ ]:
# df_panel : couverture hétérogène par pays -> irrégulier pour toutes les entités
print("Global      :", regularizer.is_regular(df_panel))
print("Par entité  :", regularizer.is_regular(df_panel, per_entity=True))

### 3.4 - Point de vigilance : entités individuellement régulières mais de fréquences différentes

`panel_mixed_native_freq` a une entité `A` (mensuelle) et une entité `B`
(trimestrielle), chacune **parfaitement régulière** prise isolément. Le
résultat par entité vaut donc `True` pour les deux, mais le résultat global
est `False` car les fréquences détectées diffèrent (`MS` ≠ `QS`) : la
condition globale n'est PAS seulement « toutes les entités sont régulières »,
elle exige en plus une fréquence commune.

In [ ]:
print("Par entité (A: MS régulière, B: QS régulière) :", regularizer.is_regular(panel_mixed_native_freq, per_entity=True))
print("Global (fréquences différentes -> False)         :", regularizer.is_regular(panel_mixed_native_freq))

### 3.5 - `panel_cols` sur un DataFrame "plat" (sans MultiIndex)

Comme pour `time_col`, `panel_cols` construit le MultiIndex à la volée à
partir de colonnes plates ; combiné à `time_col`, il reconstruit exactement
la structure `(entité, date)` attendue.

In [ ]:
df_panel_flat = df_panel.reset_index()  # 'country' et 'date' redeviennent des colonnes

print(
    "Via panel_cols=['country'], time_col='date' :",
    regularizer.is_regular(df_panel_flat, time_col='date', panel_cols=['country'], per_entity=True),
)

### 3.6 - Cas limites : moins de 2 observations

`_is_regular_ts` exige au moins 2 observations pour appeler `pd.infer_freq` ;
en-dessous, le résultat est `False` (jamais d'erreur), y compris pour une
série vide.

In [ ]:
ts_one_obs = pd.Series([1.0], index=pd.to_datetime(['2020-01-01']))
ts_empty = pd.Series([], index=pd.DatetimeIndex([]), dtype=float)

print("1 observation :", regularizer.is_regular(ts_one_obs))
print("Série vide    :", regularizer.is_regular(ts_empty))

# Panel : entité C à une seule observation -> False pour C, sans lever d'erreur pour les autres
print("panel_single_obs_entity (per_entity) :", regularizer.is_regular(panel_single_obs_entity, per_entity=True))

## 4 - `regularize(data, time_col=None, panel_cols=None, per_entity=False)`

Comble les trous de l'index en réindexant sur un `pd.date_range(start=min,
end=max, freq=...)` construit à partir de la fréquence détectée. Les lignes
ajoutées sont `NaN` sur toutes les colonnes. Aucune extrapolation au-delà de
`[min, max]` de l'index d'origine (ou, pour un panel, de `[min, max]` de
**chaque entité** — chaque entité garde sa propre couverture).

### 4.1 - Cas sain : série déjà régulière -> no-op

Quand l'index est déjà régulier, `regularize` le laisse inchangé (mêmes
longueur et valeurs).

In [ ]:
out = regularizer.regularize(ts_regular_ms)
print("Longueur avant/après :", len(ts_regular_ms), "/", len(out))
print("Valeurs identiques   :", (out == ts_regular_ms).all())

### 4.2 - Cas sain : fréquence non ancrée (`D`) avec un trou -> comblement correct

Pour une fréquence journalière — qui n'a pas de notion de position
début/fin de période — un trou est comblé correctement : la nouvelle ligne
est `NaN`, toutes les valeurs d'origine sont préservées à leur date exacte.
Ce cas sert de point de comparaison avec le bug illustré en §4.6, qui ne
touche que les fréquences *ancrées*.

In [ ]:
print("Fréquence détectée :", detect_index_frequency(ts_gap_daily.index, return_format='full'))

out = regularizer.regularize(ts_gap_daily)
print("Longueur avant/après   :", len(ts_gap_daily), "/", len(out))
print("Non-NaN avant/après     :", ts_gap_daily.notna().sum(), "/", out.notna().sum())
out

### 4.3 - Horodatages dupliqués -> `ValueError`

In [ ]:
try:
    regularizer.regularize(ts_duplicated)
except ValueError as e:
    print("ValueError :", e)

### 4.4 - Index non trié -> tri automatique

L'index est trié (`sort_index()`) avant tout traitement ; l'ordre d'entrée
n'a donc pas d'incidence sur le résultat.

In [ ]:
ts_shuffled = ts_regular_ms.sample(frac=1.0, random_state=0)  # ordre mélangé
print("Index d'entrée trié ? ", ts_shuffled.index.is_monotonic_increasing)

out = regularizer.regularize(ts_shuffled)
print("Index de sortie trié ?", out.index.is_monotonic_increasing)
print("Résultat identique à partir de l'entrée déjà triée :", out.equals(regularizer.regularize(ts_regular_ms)))

### 4.5 - `time_col` / `panel_cols` : restauration de la structure d'origine

Avec `time_col`, la colonne de date est restaurée en colonne dans le
résultat. **Point de vigilance** : pour un panel, seul le dernier niveau de
l'index (la date) est ramené en colonne — les colonnes passées via
`panel_cols` restent dans l'index, elles ne sont *pas* restaurées en
colonnes elles aussi.

In [ ]:
# Série temporelle avec time_col : restauration complète en colonne
out_ts = regularizer.regularize(df_timeseries_flat, time_col='date')
print("Colonnes (TS) :", out_ts.columns.tolist())
print(type(out_ts.index), "\n")

# Panel avec time_col + panel_cols : seule 'date' redevient une colonne, 'country' reste en index
out_panel = regularizer.regularize(df_panel_flat, time_col='date', panel_cols=['country'])
print("Colonnes (panel) :", out_panel.columns.tolist())
print("Type d'index (panel) :", type(out_panel.index), "- nom :", out_panel.index.name)
out_panel.head(3)

### 4.6 - ⚠️ BUG : fréquences ancrées (`MS`/`QS`/`YS`) + irrégularité -> perte silencieuse de toutes les valeurs

`_resolve_frequency_ts` tente d'abord `detect_index_frequency(index,
return_format='full')`. Quand `pd.Index.inferred_freq` échoue (index pas
déjà parfaitement régulier — le cas de la quasi-totalité des jeux de données
réels de ce projet), cette fonction retombe sur
`FrequencyDetector.detect_time_series_frequency`, qui peut renvoyer un code
de fréquence **sans sa position** (`M` au lieu de `MS`, `Q` au lieu de `QS`,
etc.). `regularize` construit alors `pd.date_range(start=index.min(),
end=index.max(), freq='M')` : pour une fréquence mensuelle, `'M'` (alias
déprécié de `'ME'`, fin de mois) ancre les dates en **fin** de mois, alors
que les données d'origine sont ancrées en **début** de mois (`'MS'`). Aucune
des nouvelles dates ne coïncide donc avec une date d'origine, et
`data.reindex(new_index)` renvoie un DataFrame de la bonne forme mais
**entièrement `NaN`** — sans erreur ni avertissement.

Le problème se déclenche dès qu'**un seul** point casse la régularité stricte
de l'index (cf. `ts_gap_ms` ci-dessous, qui n'a qu'un mois manquant sur 24) :
ce n'est donc pas un cas limite rare, mais un problème générique dès qu'une
fréquence ancrée rencontre la moindre irrégularité.

In [ ]:
# Un seul trou dans une série mensuelle par ailleurs propre ('MS')
print("Fréquence détectée :", detect_index_frequency(ts_gap_ms.index, return_format='full'))

out = regularizer.regularize(ts_gap_ms)
print("Longueur avant/après :", len(ts_gap_ms), "/", len(out))
print("Non-NaN avant/après   :", ts_gap_ms.notna().sum(), "/", out.notna().sum(), " <-- toutes les valeurs ont disparu")
out.head(6)

In [ ]:
# Même constat sur df_timeseries, le jeu de données RÉALISTE du notebook 3 :
# la variable annuelle démarrant avant la grille mensuelle suffit à déclencher le bug
# sur TOUTES les colonnes, y compris les colonnes purement mensuelles.
print("Non-NaN AVANT regularize :")
print(df_timeseries.notna().sum())

out_timeseries = regularizer.regularize(df_timeseries)
print("\nNon-NaN APRÈS regularize :")
print(out_timeseries.notna().sum())
out_timeseries.head(5)

In [ ]:
# Idem sur df_panel : les trois entités perdent 100% de leurs valeurs, que ce soit
# avec une fréquence globale (per_entity=False) ou détectée entité par entité (per_entity=True).
out_panel_global = regularizer.regularize(df_panel)
out_panel_entity = regularizer.regularize(df_panel, per_entity=True)

for country in df_panel.index.get_level_values('country').unique():
    n_before = df_panel.loc[country].notna().sum().sum()
    n_after_global = out_panel_global.loc[country].notna().sum().sum()
    n_after_entity = out_panel_entity.loc[country].notna().sum().sum()
    print(f"{country:12} non-NaN avant : {n_before:4d}  |  après (global) : {n_after_global:4d}  |  après (per_entity) : {n_after_entity:4d}")

In [ ]:
# Contre-exemple : sur une fréquence NON ancrée ('D', §4.2), le même mécanisme
# de repli ne perd pas la position (il n'y en a pas) et le comblement reste correct.
print("Rappel §4.2 - fréquence 'D' : non-NaN avant/après :", ts_gap_daily.notna().sum(), "/",
      regularizer.regularize(ts_gap_daily).notna().sum(), "(comblement correct)")

### 4.7 - Panel : `per_entity=True` vs `per_entity=False` (démonstration saine)

Sur un panel où chaque entité a une fréquence *native* propre et
individuellement régulière (`panel_mixed_native_freq` : `A` mensuelle, `B`
trimestrielle — cf. §3.4, aucune des deux entités ne déclenche le bug du
§4.6 puisque chacune est déjà parfaitement régulière) :
- `per_entity=True` détecte la fréquence de chaque entité indépendamment : `B`
  reste sur sa grille trimestrielle native (4 lignes, aucune ligne ajoutée).
- `per_entity=False` détecte une fréquence **globale unique** (la plus
  élevée du panel, ici `MS`) et l'applique à **toutes** les entités : `B` est
  réindexée sur la grille mensuelle, gagnant des lignes `NaN` intercalaires
  (les 4 valeurs d'origine sont conservées, car les dates trimestrielles de
  `B` — 1er janvier, avril, juillet, octobre — coïncident avec des dates de
  la grille mensuelle).

In [ ]:
print("Fréquence détectée par entité :", detect_index_frequency(panel_mixed_native_freq.index, return_format='full'))

out_true = regularizer.regularize(panel_mixed_native_freq, per_entity=True)
out_false = regularizer.regularize(panel_mixed_native_freq, per_entity=False)

print("\nper_entity=True  -> longueur(A) =", len(out_true.loc['A']), " longueur(B) =", len(out_true.loc['B']),
      " non-NaN(B) =", out_true.loc['B']['v'].notna().sum())
print("per_entity=False -> longueur(A) =", len(out_false.loc['A']), " longueur(B) =", len(out_false.loc['B']),
      " non-NaN(B) =", out_false.loc['B']['v'].notna().sum(), "(mêmes valeurs, lignes NaN en plus)")

out_false.loc['B']

### 4.8 - Panel : entité à une seule observation -> conservée telle quelle

Comme pour `is_regular` (§3.6), une entité dont la fréquence est
indétectable n'est **pas** une erreur pour `regularize` : le bloc de cette
entité est simplement conservé inchangé (`if freq is None: parts.append(data[mask]); continue`),
tandis que les autres entités sont normalement régularisées.

In [ ]:
out = regularizer.regularize(panel_single_obs_entity, per_entity=True)
print("Entité C (1 seule obs, conservée telle quelle) :")
print(out.loc['C'])
print("\nEntité A (régularisée normalement) : longueur =", len(out.loc['A']))

### 4.9 - Aucune fréquence détectable du tout -> données renvoyées inchangées

Quand ni `detect_index_frequency` ni le repli par colonnes ne parviennent à
détecter une fréquence (dates complètement irrégulières), `_resolve_frequency_ts`
renvoie `None` et `regularize` renvoie la donnée **inchangée**, sans erreur.

In [ ]:
print("Fréquence détectée :", detect_index_frequency(ts_no_freq.index, return_format='full'))

out = regularizer.regularize(ts_no_freq)
print("Longueur avant/après :", len(ts_no_freq), "/", len(out))
print("Identique à l'entrée  :", out.equals(ts_no_freq))

## 5 - Fonctions de commodité au niveau module (`is_regular`, `regularize`)

Wrappers directs autour d'une **instance singleton** (`_regularizer =
IndexRegularizer()`, définie une seule fois à l'import du module) : leur
résultat est strictement identique à celui de la méthode de classe appelée
avec les mêmes arguments — mais elles utilisent toujours
`min_observations=2` (la valeur par défaut du singleton), quel que soit le
paramètre passé à une éventuelle autre instance créée par l'appelant (cf.
§6 : ce paramètre n'a de toute façon aucun effet observable, y compris sur
le singleton lui-même).

In [ ]:
print("is_regular   identique :", is_regular(df_timeseries) == regularizer.is_regular(df_timeseries))
print("regularize   identique :", regularize(ts_gap_daily).equals(regularizer.regularize(ts_gap_daily)))

## 6 - Point de vigilance : `min_observations` est un attribut mort

`__init__` stocke `self.min_observations` et instancie un `FrequencyDetector`
interne (`self._detector`) avec cette valeur — mais ni `is_regular` ni
`regularize` ne consultent jamais `self.min_observations` ou
`self._detector` par la suite : `_is_regular_ts` compare `len(index)` à `2`
en dur, et `_resolve_frequency_ts` appelle les fonctions **de module**
`detect_index_frequency`/`detect_frequency` (qui instancient leur propre
`FrequencyDetector` interne avec ses valeurs par défaut), jamais
`self._detector`. Le paramètre `min_observations` du constructeur n'a donc
aujourd'hui **aucun effet observable**, quelle que soit sa valeur.

In [ ]:
regularizer_strict = IndexRegularizer(min_observations=50)

# Même résultat que le régularizer par défaut (min_observations=2), quelle que soit la donnée
print("is_regular identique   :", regularizer_strict.is_regular(ts_gap_ms) == regularizer.is_regular(ts_gap_ms))
print("regularize identique   :", regularizer_strict.regularize(ts_gap_daily).equals(regularizer.regularize(ts_gap_daily)))
print("\nAttribut bien stocké mais jamais lu ensuite :", regularizer_strict.min_observations, "/", regularizer_strict._detector.min_observations)

## 7 - Synthèse

| Cas | Méthode | Résultat | Comportement |
|---|---|---|---|
| TS régulière | `is_regular` | `True` | — |
| TS avec un seul trou | `is_regular` | `False` | Un trou suffit |
| `time_col` fourni | `is_regular`/`regularize` | identique à l'index déjà construit | Équivalence garantie |
| Panel, `per_entity=False` | `is_regular` | `True` ssi toutes les entités régulières **et** même fréquence | Pas seulement « toutes régulières » |
| Panel, `per_entity=True` | `is_regular` | `Dict[tuple, bool]` par entité | — |
| `panel_cols` sur DataFrame plat | `is_regular`/`regularize` | identique au MultiIndex déjà construit | — |
| < 2 observations / série vide | `is_regular` | `False` (jamais d'erreur) | — |
| TS déjà régulière | `regularize` | Inchangée (no-op) | — |
| Trou sur fréquence non ancrée (`D`) | `regularize` | Comblement correct, valeurs préservées | Cas sain |
| Horodatages dupliqués | `regularize` | `ValueError` | — |
| Index non trié | `regularize` | Trié avant traitement | Résultat indépendant de l'ordre d'entrée |
| **Trou sur fréquence ancrée (`MS`/`QS`/`YS`)** | `regularize` | **Toutes les valeurs deviennent `NaN`, sans erreur** | **⚠️ Bug (§4.6)** |
| Panel, `per_entity` (fréquences natives différentes) | `regularize` | `True`: grille propre par entité / `False`: grille globale unique | Diffère seulement si les entités ont des fréquences réellement différentes |
| Entité à une seule observation | `regularize` | Bloc conservé inchangé | Pas d'erreur |
| Aucune fréquence détectable | `regularize` | Donnée inchangée | Pas d'erreur |
| `time_col` + `panel_cols` (panel) | `regularize` | Seule la date redevient colonne | `panel_cols` reste en index |
| Fonctions de module | `is_regular`/`regularize` | Identiques à la méthode de classe | Singleton `min_observations=2` figé |
| `min_observations` (constructeur) | — | Aucun effet observable | Attribut mort (§6) |

**Points de vigilance à retenir pour les futurs tests unitaires :**
1. **Le bug du §4.6 est le point le plus important** : `regularize()` ne doit
   **jamais** être considéré comme sûr sur une fréquence ancrée (`MS`, `QS`,
   `YS`, `ME`, `QE`, `YE`, …) sans irrégularité *déjà* parfaitement résolue —
   c'est-à-dire sur la quasi-totalité des données macroéconomiques réelles du
   projet (cf. `df_timeseries`/`df_panel`). Un test de non-régression devrait
   vérifier explicitement le **nombre de valeurs non-`NaN` avant/après**, pas
   seulement la longueur du résultat (qui, elle, semble correcte et masque le
   problème).
2. `per_entity=False` sur un panel applique une fréquence **globale unique**
   (la plus élevée détectée) à toutes les entités ; seule `per_entity=True`
   respecte la fréquence native de chaque entité individuellement.
3. `panel_cols` n'est restauré en colonnes ni par `is_regular` (non
   applicable) ni par `regularize` : seul `time_col` (dernier niveau de
   l'index) l'est.
4. `min_observations` est actuellement sans effet ; à corriger ou à
   documenter comme tel si un test unitaire venait à s'y fier.